In [1]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install zipfile
!pip install pandas
!pip install opencv-python
!pip install ffmpeg-python

Looking in indexes: https://download.pytorch.org/whl/cu126
ERROR: Could not find a version that satisfies the requirement zipfile (from versions: none)
ERROR: No matching distribution found for zipfile


In [2]:
import torch
import zipfile
from google.colab import drive
from pathlib import Path
import os
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import ConcatDataset, DataLoader
import ffmpeg
import numpy as np

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [4]:

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
zip_path = '/content/drive/MyDrive/data_lenta.zip'
data_path = '/content/data_lenta'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_path)


In [6]:
# список пар (видео, csv)
dataset_pairs = [
    (Path('/content/data_lenta/Данные/43_15/43_15.mp4'),
     Path('/content/data_lenta/Данные/43_15/43_15.csv')),
    (Path('/content/data_lenta/Данные/25_12-20/25_12-20.mp4'),
     Path('/content/data_lenta/Данные/25_12-20/25_12-20.csv')),
    (Path('/content/data_lenta/Данные/26_12-20/26_12-20.mp4'),
     Path('/content/data_lenta/Данные/26_12-20/26_12-20.csv')),
]

In [7]:
def get_frame_ffmpeg(video_path, timestamp_ms, width=None, height=None):
    """
    Получаем кадр с точного timestamp в миллисекундах
    Возвращает numpy array H x W x 3 (RGB)
    """
    timestamp_s = timestamp_ms / 1000  # миллисекунды → секунды
    stream = ffmpeg.input(str(video_path), ss=timestamp_s)
    
    if width and height:
        stream = ffmpeg.filter(stream, 'scale', width, height)
    
    out, _ = (
        ffmpeg
        .output(stream, 'pipe:', format='rawvideo', pix_fmt='rgb24')
        .run(capture_stdout=True, capture_stderr=True)
    )
    
    # Получаем размеры видео
    probe = ffmpeg.probe(str(video_path))
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    w = int(video_info['width'])
    h = int(video_info['height'])
    frame = np.frombuffer(out, np.uint8).reshape([h, w, 3])
    return frame

In [8]:
class VideoYOLODatasetFFmpeg(Dataset):
    def __init__(self, video_path, csv_path, transform=None):
        self.video_path = video_path
        self.transform = transform
        
        # читаем CSV, decimal=',' для русской локали
        self.df = pd.read_csv(csv_path, decimal=',')
        self.df['filename'] = self.df['filename'].apply(lambda x: Path(x).name)
        self.df = self.df[['filename','frame_timestamp','x_min','y_min','x_max','y_max']]
        
        # группируем bbox по кадрам
        self.grouped = self.df.groupby('frame_timestamp')
        self.timestamps = list(self.grouped.groups.keys())

    def __len__(self):
        return len(self.timestamps)

    def __getitem__(self, idx):
        timestamp = self.timestamps[idx]
        frame_data = self.grouped.get_group(timestamp)
        bboxes = frame_data[['x_min','y_min','x_max','y_max']].values.tolist()
        
        # получаем кадр с точного timestamp
        frame = get_frame_ffmpeg(self.video_path, timestamp)
        
        if self.transform:
            transformed = self.transform(image=frame, bboxes=bboxes)
            frame = transformed['image']
            bboxes = transformed['bboxes']
        else:
            frame = torch.from_numpy(frame).permute(2,0,1).float() / 255.0
            bboxes = torch.tensor(bboxes, dtype=torch.float)
        
        return {"image": frame, "bboxes": torch.tensor(bboxes, dtype=torch.float)}

In [9]:
transform = A.Compose([
    A.Resize(640, 640),
    A.HorizontalFlip(p=0.5),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc'))



In [10]:
def yolo_collate_fn(batch):
    images = [item['image'] for item in batch]   # список B x C x H x W
    bboxes = [item['bboxes'] for item in batch]  # список длины B, каждый N x 4
    return {'image': images, 'bboxes': bboxes}


In [11]:

datasets = [VideoYOLODatasetFFmpeg(v, c, transform=transform) for v, c in dataset_pairs]
full_dataset = ConcatDataset(datasets)
loader = DataLoader(
    full_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=yolo_collate_fn
)

for batch in loader:
    images = batch['image']      # B x C x H x W
    bboxes = batch['bboxes']     # B x N x 4 (N - кол-во ценников на кадре)

ValueError: cannot reshape array of size 870912000 into shape (2160,3840,3)

In [ ]:
import cv2
from pathlib import Path

video_path = Path("/content/data_lenta/Данные/25_12-20/25_12-20.mp4")

# Открываем видео
cap = cv2.VideoCapture(str(video_path))

# Проверяем, удалось ли открыть
if not cap.isOpened():
    print(f"Не удалось открыть {video_path}")
else:
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration_sec = frame_count / fps if fps > 0 else 0
    print(f"Файл: {video_path.name}")
    print(f"Количество кадров: {frame_count}")
    print(f"FPS: {fps}")
    print(f"Длительность: {duration_sec:.2f} секунд")

cap.release()

Файл: 25_12-20.mp4
Количество кадров: 823
FPS: 19.577992434739407
Длительность: 42.04 секунд
